# 01 — Adaptive Exposure/Detail Exploration

Test two fixed pixel mechanisms and their ordered composition on unconsumed natural calibration groups. The historical Adaptive V3 router is not reused.

**Inputs:** Committed docs/ISSUE47_PLAN.md and compact docs/issue47 evidence; original calibration sources and pinned weights only for an explicitly authorized new pilot.

**Outputs:** Verified evidence displays by default; one opted-in acquisition writes runs/issue47/pilot and docs/issue47. No training or upstream notebook execution.

**Prerequisites:** Project kernel and committed evidence. New inference additionally requires the study environment, original BDD sources and pinned weights; the declaration and implementation must already be committed.

<a id="section-1"></a>

## 1. Configuration

Default execution reads compact committed evidence only. ALLOW_PILOT=True authorizes exactly the declared four-arm matrix on a fresh output root; any existing or incomplete attempt blocks re-execution. Read the plan before changing this flag.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from vehicle_detection.common import read_json, require_paths
from vehicle_detection.v3_pilot import run

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
require_paths(plan=ROOT / "docs/ISSUE47_PLAN.md")
ALLOW_PILOT = False

<a id="section-2"></a>

## 2. Native Paired Exploration

Four arms share original IDs/GT and JPEG95: Skip, exposure component, detail component, and the full routed treatment. These are pretrained natural calibration results, not S20 detector results. Subgroup AP covers represented classes only; missing rare-class support blocks advancement.

In [ ]:
result = run(ROOT, allow_inference=ALLOW_PILOT)
display(pd.DataFrame(result["native"]).T)
display(pd.DataFrame(result["runtime"]).T)
display(result["routing"])
for subgroup in (
    "exposure",
    "detail",
    "both",
    "skip",
    "dark",
    "bright",
    "blurred",
    "small_two_wheeler",
):
    print(subgroup)
    display({arm: records[subgroup] for arm, records in result["metrics"].items()})
display(Image(filename=str(ROOT / "docs/issue47/pilot.png")))

<a id="section-3"></a>

## 3. Errors and Uncertainty

Fixed-confidence .25 / IoU .50 TP/FP counts are illustrative and never per-image AP. The 100-draw exploratory interval resamples original source groups, records missing classes, and has no multiplicity correction. The saved examples use TP loss then FP rise, with ID tie-breaking.

In [ ]:
report = ROOT / "docs/issue47"
uncertainty = read_json(report / "uncertainty.json")
display(
    {
        key: uncertainty[key]
        for key in (
            "recovery",
            "recovery_ci_low",
            "recovery_ci_high",
            "valid_draws",
            "invalid_draws",
            "estimand",
        )
    }
)
errors = read_json(report / "errors.json")
for subgroup in ("dark", "blurred", "small_two_wheeler"):
    print(subgroup)
    display({i: errors["counts"][i] for i in errors["examples"][subgroup]})
display(Image(filename=str(report / "errors.png")))

<a id="section-4"></a>

## 4. Exploration Summary

Apply the preregistered gates once. Failed or unsupported routes stop the complete candidate; components cannot become post hoc replacements. Confirmation, preparation, training and development evaluation require a defensible frozen policy and separate saved prerequisites. Cross-version presentation belongs exclusively in ../compare.ipynb.

In [ ]:
display(result["decision"])
print("No policy is promoted by this notebook. See docs/ISSUE47_RESULTS.md for the decision.")
print("Confirmation, development and final pixels stay sealed during this pilot.")